# CoAx quickstart: watch backup heads wake up

This run-all notebook follows the core CoAx story on GPT-2-small IOI: measure heads in the intact model, remove the supplied primary name-mover circuit, and rank the heads whose causal effect grows. It uses a reduced 8-prompt setting for an accessible demo; use the repository's `make headline` target for the paper protocol.

## 1. Set up the repository

The cell reuses a local checkout when one is available and otherwise clones the public repository. Model weights are downloaded and cached by Hugging Face on first use.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file()), None)
if root is None:
    root = Path.cwd() / 'Conditional-Co-Ablation'
    if not root.exists():
        subprocess.run([
            'git', 'clone', '--depth', '1',
            'https://github.com/GongZhiren/Conditional-Co-Ablation.git', str(root)
        ], check=True)
os.chdir(root)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.', 'matplotlib', 'pandas'], check=True)
print(f'Repository: {Path.cwd()}')

## 2. Run the forward-only CoAx demo

The documented primary heads are supplied to CoAx. Backup labels are not used to compute or rank scores; they are loaded only afterward to evaluate and annotate the ranking.

In [ ]:
command = [
    sys.executable, 'experiments/paper/backup_recovery_full.py',
    '--model-key', 'gpt2-small',
    '--num-prompts', '8',
    '--seeds', '1',
    '--position-mode', 'last',
    '--top-r', '192',
    '--skip-grad',
    '--suffix', '_quickstart',
]
run_env = os.environ.copy()
run_env.setdefault('CUDA_VISIBLE_DEVICES', '0')  # keep this small demo on one GPU
subprocess.run(command, check=True, env=run_env)

## 3. Inspect the recovered backups

The table separates the intact-state score, the removed-state control, and their conditional difference (CoAx). Positive CoAx scores identify heads whose intervention effect increases after primary removal.

In [ ]:
import json
import numpy as np
import pandas as pd

artifact = json.loads(Path('outputs/coablation/bc_dump_seed1_quickstart.json').read_text())
scores = artifact['scores']
single_key = 'single-ablation saliency (1st-order)'
conditional_key = 'conditional energy (removed-state control)'
coax_key = 'conditional co-ablation (ours, 2nd-order)'
candidates = np.asarray(scores['cand'], dtype=int)
labels = np.asarray(scores['y'], dtype=bool)
coax = np.asarray(scores[coax_key], dtype=float)
order = np.argsort(-coax)[:8]

top_heads = pd.DataFrame({
    'head': [f'L{u // 12}H{u % 12}' for u in candidates[order]],
    'intact score': np.asarray(scores[single_key])[order],
    'removed-state score': np.asarray(scores[conditional_key])[order],
    'CoAx': coax[order],
    'documented backup': labels[order],
})
display(top_heads.style.format({
    'intact score': '{:.4f}', 'removed-state score': '{:.4f}', 'CoAx': '{:+.4f}'
}).background_gradient(subset=['CoAx'], cmap='YlOrRd'))

pd.DataFrame.from_dict(artifact['backup_recovery_auc'], orient='index', columns=['backup ROC-AUC'])

## 4. Visualize the wake-up pattern

Each cell is one attention head. Gray cells are the supplied primary heads, which are excluded from candidate ranking. The CoAx panel highlights conditional increases rather than large intact-state effects.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), constrained_layout=True)
panels = [
    ('Intact-state effect', single_key, 'viridis'),
    ('Primary removed', conditional_key, 'viridis'),
    ('CoAx: conditional increase', coax_key, 'coolwarm'),
]
for ax, (title, key, cmap_name) in zip(axes, panels):
    grid = np.full(12 * 12, np.nan)
    grid[candidates] = np.asarray(scores[key], dtype=float)
    cmap = plt.get_cmap(cmap_name).copy()
    cmap.set_bad('#d9d9d9')
    image = ax.imshow(grid.reshape(12, 12), cmap=cmap, aspect='equal')
    ax.set(title=title, xlabel='Head', ylabel='Layer', xticks=range(0, 12, 2), yticks=range(0, 12, 2))
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
plt.show()

## Next steps

- Run `make headline` for the complete four-seed, 96-prompt, full-vocabulary comparison.
- Run `make mechanism` and `make causal` to test graded hand-off and freezing.
- Run `make completion` and `make knockout` to validate the completed circuit causally.

See the README and `REPRODUCIBILITY.md` for paper-scale commands, expected artifacts, model requirements, and protocol notes.